# Exploratory Data Analysis (EDA) Project
## Exploring Student Academic Performance

**Objective:** Explore a dataset to discover patterns, distributions, relationships, and possible factors associated with students' exam scores.

This notebook uses a sample dataset generated in Python, so no external CSV download is needed. It includes statistical summaries, missing-data checks, visualizations, correlation analysis, group comparisons, and a structured report.

**Important:** The data is synthetic for educational practice. Relationships found in this notebook are associations, not proof of cause and effect.

## 1. Import libraries

We use Pandas and NumPy to work with data, and Matplotlib and Seaborn to visualize it.

In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")
print("Libraries imported successfully.")

## 2. Generate a sample dataset

Each row represents a student. Features include study hours, attendance, sleep, prior score, participation, and final exam score. We introduce a few missing values and duplicate rows so that we can practice checking data quality.

In [ ]:
rng = np.random.default_rng(42)
n = 250

study_hours = rng.uniform(0.5, 8, n)
attendance = rng.uniform(55, 100, n)
sleep_hours = np.clip(rng.normal(7, 1.1, n), 3.5, 10)
prior_score = np.clip(rng.normal(68, 12, n), 25, 100)
participation = rng.integers(1, 11, n)

# Generate a plausible score with noise; this is only synthetic example data.
noise = rng.normal(0, 7, n)
exam_score = (
    22
    + 4.2 * study_hours
    + 0.28 * attendance
    + 0.22 * prior_score
    + 1.0 * participation
    + 1.2 * sleep_hours
    + noise
)
exam_score = np.clip(exam_score, 0, 100).round(1)

df = pd.DataFrame({
    "Student_ID": [f"STU{i:03d}" for i in range(1, n + 1)],
    "Study_Hours_Per_Day": study_hours.round(1),
    "Attendance_Percent": attendance.round(1),
    "Sleep_Hours": sleep_hours.round(1),
    "Prior_Exam_Score": prior_score.round(1),
    "Class_Participation_Score": participation,
    "Final_Exam_Score": exam_score
})

# Add intentional missing values and duplicate rows for data-quality practice.
for col, count in [("Study_Hours_Per_Day", 5), ("Attendance_Percent", 4),
                   ("Sleep_Hours", 4), ("Final_Exam_Score", 3)]:
    missing_idx = rng.choice(df.index, size=count, replace=False)
    df.loc[missing_idx, col] = np.nan

df = pd.concat([df, df.iloc[[10, 35, 80]]], ignore_index=True)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head(10))

## 3. Inspect the dataset structure

Check row and column counts, column names, data types, and a sample of records.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nFirst five rows:")
display(df.head())

## 4. Statistical summaries

Descriptive statistics help us understand typical values, spread, and ranges. For example, the mean and median show the center of a distribution, while standard deviation indicates how spread out values are.

In [ ]:
display(df.describe(include="all").T)

numeric_columns = df.select_dtypes(include=np.number).columns
summary = pd.DataFrame({
    "Mean": df[numeric_columns].mean(),
    "Median": df[numeric_columns].median(),
    "Std_Dev": df[numeric_columns].std(),
    "Min": df[numeric_columns].min(),
    "Max": df[numeric_columns].max()
})
display(summary.round(2))

## 5. Data-quality checks

Inspect missing values and duplicate records. We will keep the original `df` unchanged and create a cleaned copy in the next section.

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
display(missing_counts[missing_counts > 0].to_frame("Missing_Count"))

duplicate_count = int(df.duplicated().sum())
print(f"\nExact duplicate rows: {duplicate_count}")

plt.figure(figsize=(9, 4.5))
missing_counts.sort_values(ascending=False).plot(kind="bar")
plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Number of missing cells")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 6. Prepare a clean dataset for analysis

For this demonstration, remove exact duplicate rows and fill missing numeric values with each column's median. Median imputation is a simple baseline; in real work, choose a method based on the meaning of the data and document it.

In [ ]:
clean = df.drop_duplicates().copy()

numeric_cols = clean.select_dtypes(include=np.number).columns
for col in numeric_cols:
    clean[col] = clean[col].fillna(clean[col].median())

print("Original shape:", df.shape)
print("Cleaned shape:", clean.shape)
print("Missing cells remaining:", int(clean.isna().sum().sum()))
print("Duplicate rows remaining:", int(clean.duplicated().sum()))

display(clean.head())

## 7. Univariate analysis: examine one variable at a time

Histograms show distributions, while boxplots help compare spread and identify values that may deserve closer inspection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(data=clean, x="Final_Exam_Score", bins=18, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Final Exam Scores")
axes[0].set_xlabel("Final exam score")

sns.boxplot(data=clean, y="Study_Hours_Per_Day", ax=axes[1])
axes[1].set_title("Study Hours: Spread and Potential Outliers")
axes[1].set_ylabel("Study hours per day")

plt.tight_layout()
plt.show()

## 8. Bivariate analysis: compare two variables

A scatterplot helps us inspect whether study time and exam scores move together. The trend line summarizes the linear pattern, but it does not prove that study hours alone cause a score change.

In [ ]:
plt.figure(figsize=(8, 5.5))
sns.regplot(
    data=clean, x="Study_Hours_Per_Day", y="Final_Exam_Score",
    scatter_kws={"alpha": 0.55}, line_kws={"color": "black"}
)
plt.title("Study Hours vs. Final Exam Score")
plt.xlabel("Study hours per day")
plt.ylabel("Final exam score")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

sns.scatterplot(data=clean, x="Attendance_Percent", y="Final_Exam_Score", alpha=0.65, ax=axes[0])
axes[0].set_title("Attendance vs. Final Exam Score")
axes[0].set_xlabel("Attendance (%)")
axes[0].set_ylabel("Final exam score")

sns.scatterplot(data=clean, x="Sleep_Hours", y="Final_Exam_Score", alpha=0.65, ax=axes[1])
axes[1].set_title("Sleep Hours vs. Final Exam Score")
axes[1].set_xlabel("Sleep hours")
axes[1].set_ylabel("Final exam score")

plt.tight_layout()
plt.show()

## 9. Compare groups

Create study-hour groups to compare average exam scores. Grouping can make a pattern easier to communicate, but it also simplifies a continuous variable and may hide variation within each group.

In [ ]:
clean["Study_Group"] = pd.cut(
    clean["Study_Hours_Per_Day"],
    bins=[0, 2, 4, 6, 8],
    labels=["0–2 hours", "2–4 hours", "4–6 hours", "6–8 hours"],
    include_lowest=True
)

group_summary = clean.groupby("Study_Group", observed=False)["Final_Exam_Score"].agg(
    ["count", "mean", "median", "std"]
).round(2)
display(group_summary)

plt.figure(figsize=(9, 5))
sns.barplot(data=clean, x="Study_Group", y="Final_Exam_Score", errorbar=None)
plt.title("Average Exam Score by Study-Hour Group")
plt.xlabel("Study hours per day")
plt.ylabel("Average final exam score")
plt.tight_layout()
plt.show()

## 10. Correlation analysis

Pearson correlation measures the strength and direction of a linear relationship between numeric variables. Values near +1 indicate a positive linear association, values near −1 a negative one, and values near 0 little linear association. Correlation does not establish causation.

In [ ]:
corr_cols = [
    "Study_Hours_Per_Day", "Attendance_Percent", "Sleep_Hours",
    "Prior_Exam_Score", "Class_Participation_Score", "Final_Exam_Score"
]
corr = clean[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

display(corr["Final_Exam_Score"].drop("Final_Exam_Score").sort_values(ascending=False).to_frame("Correlation_with_Final_Score"))

## 11. Identify the strongest associations with the outcome

Rank the numeric features by their absolute correlation with final exam score. This is a quick exploratory check, not a causal or definitive measure of importance.

In [ ]:
score_correlations = corr["Final_Exam_Score"].drop("Final_Exam_Score")
ranked_correlations = score_correlations.reindex(
    score_correlations.abs().sort_values(ascending=False).index
)
display(ranked_correlations.to_frame("Correlation_with_Final_Score"))

strongest_feature = ranked_correlations.abs().idxmax()
strongest_value = ranked_correlations[strongest_feature]
direction = "positive" if strongest_value > 0 else "negative"
print(
    f"The strongest absolute linear association with final exam score in this sample "
    f"is {strongest_feature} (r = {strongest_value:.2f}, {direction})."
)

## 12. Structured EDA report

This section generates a concise report using the calculated statistics. Because the dataset is synthetic, use it to learn how to structure an EDA summary rather than to make real claims about students.

In [ ]:
mean_score = clean["Final_Exam_Score"].mean()
median_score = clean["Final_Exam_Score"].median()
std_score = clean["Final_Exam_Score"].std()
mean_study = clean["Study_Hours_Per_Day"].mean()
mean_attendance = clean["Attendance_Percent"].mean()
top_group = group_summary["mean"].idxmax()
top_group_score = group_summary.loc[top_group, "mean"]

print("EXPLORATORY DATA ANALYSIS REPORT")
print("=" * 55)
print(f"1. Dataset: {len(clean)} records after removing exact duplicates.")
print(f"2. Final exam scores: mean = {mean_score:.1f}, median = {median_score:.1f}, standard deviation = {std_score:.1f}.")
print(f"3. Typical study time: approximately {mean_study:.1f} hours per day on average.")
print(f"4. Average attendance: approximately {mean_attendance:.1f}%.")
print(f"5. Study-hour group with the highest average score: {top_group} (mean score = {top_group_score:.1f}).")
print(f"6. Strongest absolute linear association with final score: {strongest_feature} (r = {strongest_value:.2f}).")
print("\nCautions:")
print("- The dataset is synthetic and may reflect patterns built into its generation.")
print("- Correlation does not imply causation.")
print("- Missing-value imputation and duplicate removal may affect summaries.")
print("\nSuggested next steps:")
print("- Validate patterns using real, representative data.")
print("- Explore whether patterns differ across relevant groups.")
print("- Investigate unusual observations and potential confounding variables.")

## 13. Optional: export the cleaned dataset

The cleaned data can be saved as a CSV file in the notebook's current working directory.

In [ ]:
clean.to_csv("cleaned_student_performance_eda.csv", index=False)
print("Saved: cleaned_student_performance_eda.csv")

## Conclusion

This EDA workflow moves from understanding the dataset to checking data quality, describing individual variables, exploring relationships, comparing groups, examining correlations, and communicating findings in a structured report.

**Core lesson:** exploratory analysis helps generate questions and hypotheses. Confirm important findings with appropriate methods and domain knowledge before making decisions.